In [7]:
import os
from datasets import load_dataset, Dataset
from functools import partial

# 1. Configuration
SOURCE_DATASET = "nordolemil/crawlzilla"
OUTPUT_DIR = "./crawlzilla_shuffled_local"  # Path on your hard drive
BUFFER_SIZE = 500_000  # Number of rows to hold in RAM for shuffling
SEED = 42

def gen_from_iterable_dataset(iterable_ds):
    """Generator function to yield rows from the shuffled stream."""
    for example in iterable_ds:
        yield example

def main():
    print(f"Connecting to {SOURCE_DATASET} stream...")
    
    # 2. Load the dataset in streaming mode
    # No data is downloaded yet.
    streamed_ds = load_dataset(SOURCE_DATASET, streaming=True, split="train")

    # 3. Apply the shuffle with a large buffer
    # This provides a 'local' shuffle as the data streams in.
    shuffled_stream = streamed_ds.shuffle(seed=SEED, buffer_size=BUFFER_SIZE)

    print(f"Starting shuffle and save process to: {OUTPUT_DIR}")
    print("This will process the dataset shard-by-shard to save disk/RAM.")

    # 4. Use from_generator to write to disk
    # This pulls from the shuffled stream and writes directly to local Arrow files.
    # It handles OOM (Out of Memory) by streaming the write process.
    local_dataset = Dataset.from_generator(
        partial(gen_from_iterable_dataset, shuffled_stream),
        features=streamed_ds.features,
        cache_dir=OUTPUT_DIR
    )

    # 5. Finalize the format on disk
    local_dataset.save_to_disk(OUTPUT_DIR)

    print(f"\nSuccess! Shuffled dataset saved to {OUTPUT_DIR}")
    print("You can now manually upload this folder to Hugging Face.")

if __name__ == "__main__":
    main()

Connecting to nordolemil/crawlzilla stream...
Starting shuffle and save process to: ./crawlzilla_shuffled_local
This will process the dataset shard-by-shard to save disk/RAM.


Generating train split: 7582862 examples [18:43, 15678.51 examples/s]'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: c5257bfe-d72a-4872-bf76-45f6a7f1d980)')' thrown while requesting GET https://huggingface.co/datasets/nordolemil/crawlzilla/resolve/f3d367972e619554bf0d6cb5f30a772ee3e35661/part-00030.parquet
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 6395e5a7-1375-427b-bb2e-1ab9a178821d)')' thrown while requesting GET https://huggingface.co/datasets/nordolemil/crawlzilla/resolve/f3d367972e619554bf0d6cb5f30a772ee3e35661/part-00030.parquet
Retrying in 2s [Retry 2/5].
Generating train split: 17084238 examples [42:23, 15658.24 examples/s]'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 7ef9e1c7-a418-4e2c-a161-a128fd656697)')' thrown while 


Success! Shuffled dataset saved to ./crawlzilla_shuffled_local
You can now manually upload this folder to Hugging Face.
